# Network Analysis of "Badshad" case

## set up environment

Load the library dependencies

In [1]:
import json
import pathlib
import traceback
import typing

from icecream import ic
import networkx as nx
import polars as pl
import pyvis
import watermark

Show a watermark of the OS, hardware, language environment, and dependent library versions.

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2026-05-15T17:10:40.196891-07:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.13.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.4.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

json     : 2.0.9
networkx : 3.6.1
polars   : 1.40.1
pyvis    : 0.3.2
watermark: 2.6.0



## load the OpenSanctions dataset

In [10]:
data_path: pathlib.Path = pathlib.Path("data")

In [4]:
def extract_os_rec (
    dat: dict,
    ) -> dict:
    "extract a JSONL record, to reshape the data as a Polars dataframe"
    rec: dict = {
        "record_type": dat["RECORD_TYPE"],
        "record_id": dat["RECORD_ID"],
        "data_source": dat["DATA_SOURCE"],
    }

    # extract the full name
    if "NAMES" in dat:
        if "NAME_FULL" in dat["NAMES"][0]:
            rec["name"] = dat["NAMES"][0]["NAME_FULL"]
        elif "NAME_ORG" in dat["NAMES"][0]:
            rec["name"] = dat["NAMES"][0]["NAME_ORG"]
        else:
            print(dat["NAMES"])

    # extract the full address
    if "ADDRESSES" in dat and len(dat["ADDRESSES"]) > 0:
        if "ADDR_FULL" in dat["ADDRESSES"][0]:
            rec["address"] = dat["ADDRESSES"][0]["ADDR_FULL"];

    # extract the country code
    if "COUNTRIES" in dat:
        c_dict: dict = dict([list(x.items())[0] for x in dat["COUNTRIES"]])

        if "REGISTRATION_COUNTRY" in c_dict:
            rec["country"] = c_dict["REGISTRATION_COUNTRY"]
        elif "NATIONALITY" in c_dict:
            rec["country"] = c_dict["NATIONALITY"]
        elif "CITIZENSHIP" in c_dict:
            rec["country"] = c_dict["CITIZENSHIP"]
        else:
            print(c_dict)

    # extract the risks, if any
    if "RISKS" in dat:
        rec["risks"] = [item["TOPIC"] for item in dat["RISKS"]]
    else:
        rec["risks"] = []

    # extract relationships, if any
    if "RELATIONSHIPS" in dat:
        rec["links"] = []

        for item in dat["RELATIONSHIPS"]:
            if "REL_POINTER_KEY" in item:
                rec["links"].append((
                    item["REL_POINTER_ROLE"],
                    item["REL_POINTER_KEY"],
                ))

    return rec

In [5]:
extracted_dat: list = []
os_file: pathlib.Path = data_path / "open-sanctions.json"

with os_file.open() as fp:
    for line in fp:
        dat: dict = json.loads(line.strip())

        try:
            extracted_dat.append(extract_os_rec(dat))
        except Exception as ex:
            print(ex)
            print(traceback.format_exc())
            break

df: pl.DataFrame = pl.DataFrame(extracted_dat)

In [6]:
df.head()

record_type,record_id,data_source,name,address,country,risks,links
str,str,str,str,str,str,list[str],list[list[str]]
"""PERSON""","""NK-25vyVFzt8vdJGgAXMRTwTJ""","""OPEN-SANCTIONS""","""Abassin BADSHAH""","""31 Quernmore Close, Bromley, K…","""gb""","[""corp.disqual""]","[[""Directorship"", ""NK-SKAADAiqiZ78JsJjeg72Te""], [""Directorship"", ""NK-3p3mmVWmjwVtTfKchz4kNE""]]"
"""ORGANIZATION""","""NK-3p3mmVWmjwVtTfKchz4kNE""","""OPEN-SANCTIONS""","""LMAR (GB) LTD""","""31 Quernmore Close, Bromley, K…","""gb""",[],[]
"""ORGANIZATION""","""NK-auyPsLrBzRoxjCRWgjBvas""","""OPEN-SANCTIONS""","""WANDLE HOLDINGS LIMITED""","""DEANA BEACH APTS, BLOCK A, Fla…","""cy""","[""sanction.linked""]",[]
"""ORGANIZATION""","""NK-cf4Q3KcmUnQbt8Cy7iTtwK""","""OPEN-SANCTIONS""","""POLYUS GOLD INTERNATIONAL LIMI…","""3RD FLOOR CHARTER PLACE 23-27 …","""je""","[""sanction.linked""]",[]
"""PERSON""","""NK-dNNN56A4ApVfUFvfzniLCF""","""OPEN-SANCTIONS""","""Firuza Nazimovna Kerimova""","""MOSCOW, RUS, 123430""","""ru""","[""role.rca"", ""sanction""]","[[""Family"", ""Q447250""]]"


## network visualization

In [7]:
role_colors: dict = {
    "source": "#BDC561",
    "multi-source": "#D9CAD7",
    "drain": "#C25FB8",
}

role_shapes: dict = {
    "source": "square",
    "multi-source": "diamond",
    "drain": "circle",
}

In [8]:
def render_graph (
    graph: nx.DiGraph,
    ) -> pyvis.network.Network:
    """
    Generate a `PyVis` network visualization from the DiGraph
    """
    net: pyvis.network.Network = pyvis.network.Network(
        notebook = True,
        cdn_resources = "in_line",
        directed = True,
        height = "900px", 
        width = "100%",
    )
    
    # adjust node attributes
    net.from_nx(graph)

    for node in net.nodes:
        node["value"] = node["volume"]
        node["label"] = node["name"]

        node["color"] = role_colors[node["role"]]
        node["shape"] = role_shapes[node["role"]]

        if node["name"] == "BAKTELEKOM MMC":
            node["color"] = "#3381EA"

        title: str = node["name"] + "\nnode_id: " + str(node["id"])

        for key in [ "role", "rank", "count" ]:
            title += f"\n{key}: " + str(node[key])

        for key in [ "volume", "balance", "credits", "debits", ]:
            title += f"\n{key}: {node[key]:.3e}"
        
        node["title"] = title
    
    # adjust edge attributes
    for edge in net.edges:
        arc: list = sorted([ edge["from"], edge["to"] ])

        if arc in strong_edges:
            edge["width"] = 10

    return net

construct a graph `PyVis` for visualization

In [9]:
net: pyvis.network.Network = render_graph(graph)

net.force_atlas_2based()
net.toggle_physics(True)

net.prep_notebook()
net.show_buttons(filter_ = [ "physics" ])

options: str =     """
  "physics": {
    "forceAtlas2Based": {
      "centralGravity": 0.04,
      "springLength": 5,
      "springConstant": 0.32,
      "avoidOverlap": 0.02
    },
    "minVelocity": 0.75,
    "solver": "forceAtlas2Based"
  }
    """

#net.set_options(options)
# setting options via Py interface doesn't work?

net.show("aml.html")

NameError: name 'graph' is not defined